# CityOps Copilot - Agent memory workshop

Built on Oracle AI Database + the `oracleagentmemory` SDK, seeded with ~220 inspection findings across 26 urban infrastructure assets (bridges, substations, pipelines, water treatment, sensors, comms towers, seawalls), plus realistic maintenance narratives that the SDK distills into memory. OCI Generative AI supplies both the LLM (`openai.gpt-oss-120b`) and embedding model (`cohere.embed-v4.0`).


A field assistant for the inspectors who keep a city's infrastructure safe - bridges, substations, pipelines, water-treatment plants, seawalls, and more. When an inspector is on site and writes up what they see, the copilot interprets the narrative, suggests a likely diagnosis, and points to the relevant history for that asset - so the inspector reasons from the full record, not just today's visit.

What makes it more than a chatbot is **memory**. It retains every asset's inspection history - across inspectors and across visits. So when a new inspector encounters a corrosion concern on Harbor Bridge, the copilot already knows what the prior inspector observed, recommended, and graded - without anyone telling it.


>  Open `docs/overview.md` for the one-pager.


## The data & memory layer

The copilot stands on **two kinds of state**, and telling them apart is the heart of the design:

- **Domain data** the agent *reads* - the facts of the physical world, stored in two plain SQL tables we load ourselves.
- **Memory** the agent *forms* - what inspectors said and what was learned from it, stored and grown by the SDK.

### 1 · Domain data - the system of record

The copilot reads these tables; it never invents their contents. Both are ordinary Oracle tables, bulk-loaded from the committed datasets - and `CITY_INSPECTION_FINDING` also grows as inspectors record new findings (`log_finding`).

**`CITY_ASSET`** - the asset registry. One row per piece of infrastructure.

| asset_id | asset_class |
|---|---|
| `Harbor Bridge` | `bridge` |

**`CITY_INSPECTION_FINDING`** - structured findings, one row per observation, with a `VECTOR(OCI_EMBED_DIM)` column for semantic search.

| finding_id | asset_id | inspector | grade | category | severity | description | recommendation | embedding |
|---|---|---|---|---|---|---|---|---|
| `a1b2c3d4e5f6` | `Harbor Bridge` | `Evelyn H. Mercer` | `B` | `corrosion` | `medium` | *"Surface corrosion and pitting on steel bearing assemblies at Pier 2; ~25% section loss…"* | *"Remove loose corrosion products, apply inhibiting primer within 60 days…"* | `[0.021, -0.044, ...]` |

### 2 · Memory - what the agent accumulates (SDK)

This is state the agent *forms* from interaction, not data we load. Unlike the domain tables, the SDK creates and owns both memory tables - `CITY_MESSAGE` and `CITY_MEMORY` - for you; you never write their DDL. It all comes from one call - `thread.add_messages(...)` - which produces **two distinct things**:

- **Conversational history** → stored verbatim in `CITY_MESSAGE`, scoped to that asset's thread. The raw record of *what each inspector said*, in order. **Thread-local.**
- **Extracted records** → the SDK's extractor LLM reads each narrative and distills typed, reusable memories into `CITY_MEMORY`:
  - **`fact`** - durable realities (*"Pier 2 bearings show recurring section loss"*)
  - **`preference`** - stable choices or styles an inspector defaults to
  - **`guideline`** - reusable rules and "next time do X" lessons
  
  These are **reusable across threads and inspectors** - which is exactly why a *new* inspector benefits from a *prior* inspector's experience.


## Design

Every copilot turn is a **read-then-write cycle** over a single Oracle connection: pull context from the data and memory layers, ask the LLM, then persist - and that write is what grows memory for next time.

### The big picture

A copilot turn reads across both layers but **only writes to memory** - it never writes the domain tables. Every write flows through one `add_messages()`: the turn lands in `CITY_MESSAGE`, and the SDK derives the rest (dashed arrows) - distilling memory into `CITY_MEMORY` and refreshing the running summary on `CITY_THREAD`.

![CityOps architecture: the copilot reads from the domain and SDK-memory stores, and writes are deliberate (log_finding) vs automatic (add_messages)](diagrams/big-picture.svg)

### A copilot turn, end to end

The reads assemble the prompt; the LLM answers; the write closes the loop by feeding the next turn. The **same step numbers** appear in the diagram and in the worked example below, so you can read them side by side.

![A copilot turn end to end: the eight steps of call_copilot, from resolving the thread to deriving memory](diagrams/copilot-turn.svg)

#### Worked example - the same turn, step by step

Say an inspector arrives at **Harbor Bridge** - an asset a *different* inspector visited weeks earlier - and types:

> *"Rust bleed near a pier on the south side, and what looks like spalling on the concrete pedestal below."*

Each row is the **same numbered step** from the diagram, with the concrete value it produces for this turn:

| # | Step | What flows for this turn |
|:--:|---|---|
| 1 | `resolve thread` | get-or-create the `asset_harbor_bridge` thread for this inspector |
| 2 | `get_asset()` | `Harbor Bridge` → class `bridge` |
| 3 | `get_context_card()` | the asset's earlier messages + extracted facts - e.g. a prior note to *coordinate remediation with the Q3 deck resurfacing* |
| 4 | `find_similar_findings()` | top hit: *Pier 2 bearing corrosion, ~25% section loss, grade `C`, "apply inhibiting primer within 60 days"* |
| 5 | `prompt → LLM` | system prompt + the assembled context (asset record + context card + similar findings + the new narrative) |
| 6 | `diagnosis` | the LLM ties the new rust bleed → the documented Pier 2 corrosion; cites the prior grade and recommendation |
| 7 | `add_messages()` | the narrative + the answer are written verbatim to `CITY_MESSAGE` |
| 8 | `extractor` | distills any new `fact` from this turn into `CITY_MEMORY` for next time |

The new inspector's first-ever look at this bridge is already shaped by an earlier inspection - **no human handoff**. That's the read path (steps 2–4) consuming exactly what an earlier write path (steps 7–8) produced.

### How it ties together

Every answer reads from **both** layers at once: the system of record supplies the facts (the asset and its logged findings), and memory supplies the context (recent messages, distilled records, the running summary). What separates them is *how they grow* - a finding is recorded with `log_finding`, while memory accrues **automatically** from each `add_messages`. Both feed the next turn's reads, so a later inspection - even by a different inspector at the same asset - opens already aware of what came before. That's the feedback loop the rest of the workshop builds.

### How we'll build it

You won't build the copilot all at once. Parts 3–5 each **build one component and verify it in isolation**; Part 6 **assembles** them into `call_copilot`. Nothing here is throwaway - every piece is wired into the final turn.

| Part | Component you build | Proven by | Powers in `call_copilot` |
|---|---|---|---|
| **3** | Conversational memory + auto-extraction (`report_event`) | TODO 1–2 checkpoints | the context card + the write-back step |
| **4** | Structured findings + vector search (`log_finding`, `find_similar_findings`) | TODO 3 checkpoint | the "similar past findings" retrieval |
| **5** | Scoping (user / agent / thread) | scope assertions | safe reads across many inspectors |
| **6** | **Assemble** all three into `call_copilot` | end-to-end smoke test | - |


# Part 1: Oracle setup



Oracle AI Database is reachable at the configured DSN. We use the same basic
connect pattern as the `rag_to_agents` lab, but the first code cell also loads
`.env` values when present so OCI Generative AI credentials can be supplied to
the notebook kernel.

The SDK creates its tables under prefix `CITY_` in Part 2.


In [ ]:
# Display helpers (shared style with the rag_to_agents lab).
# ok() prints a green check so you can tell at a glance that a cell finished,
# even after the In [*] marker scrolls off-screen.
import json
import html
import decimal
import os
import time
from pathlib import Path
from contextlib import contextmanager
from typing import Any, Iterable
from IPython.display import HTML, display
from dotenv import load_dotenv


def ok(msg: str) -> None:
    """Render a green check + message as cell output."""
    display(HTML(
        f"<span style='color:#1a7f37;font-weight:700'>&#10003;</span>"
        f" <span style='color:#1a7f37'>{html.escape(msg)}</span>"
    ))


def _json_default(obj: Any) -> Any:
    if isinstance(obj, decimal.Decimal):
        return float(obj)
    if hasattr(obj, "isoformat"):
        return obj.isoformat()
    raise TypeError(f"Not JSON-serializable: {type(obj).__name__}")


def show_table(headers: list, rows: Iterable, max_width: int = 80) -> None:
    """Render a rowset as a simple HTML table."""
    def cell(v: Any) -> str:
        s = "" if v is None else str(v)
        if len(s) > max_width:
            s = s[: max_width - 1] + "\u2026"
        return html.escape(s)
    thead = "".join(
        f"<th style='text-align:left;padding:4px 10px;border-bottom:1px solid #ccc'>{html.escape(h)}</th>"
        for h in headers
    )
    body = []
    for r in rows:
        tds = "".join(
            f"<td style='padding:4px 10px;border-bottom:1px solid #eee;vertical-align:top'>{cell(v)}</td>"
            for v in r
        )
        body.append(f"<tr>{tds}</tr>")
    display(HTML(f"<table style='border-collapse:collapse'>{thead}{''.join(body)}</table>"))


def print_json(result: Any) -> None:
    """Pretty-print a JSON-ish result, handling Oracle Decimal and dates."""
    if isinstance(result, (bytes, bytearray)):
        result = result.decode("utf-8")
    if isinstance(result, str):
        try:
            result = json.loads(result)
        except json.JSONDecodeError:
            print(result)
            return
    print(json.dumps(result, indent=2, default=_json_default))


@contextmanager
def timed(label: str):
    """Print elapsed wall-clock time for a notebook step."""
    start = time.perf_counter()
    try:
        yield
    finally:
        print(f"{label}: {time.perf_counter() - start:.2f}s")


print("Helpers loaded: ok, show_table, print_json, timed")

# Load .env if the notebook kernel was not started from a shell that already
# exported the workshop variables.
load_dotenv()

# === ORACLE AI DATABASE ===
DB_USER     = os.getenv("DBUSER", "prism")
DB_PASSWORD = os.getenv("DBPASSWORD", os.getenv("PASSWORD", "CHANGE_ME"))
DB_DSN      = os.getenv("DB_DSN", "aidbfree:1521/FREEPDB1")

# === OCI GENERATIVE AI - API KEY CONFIG FILE AUTH ===
# Required: OCI_COMPARTMENT_ID in the environment or .env.
# API key credentials come from OCI_CONFIG_FILE / OCI_PROFILE, defaulting to
# ~/.oci/config and DEFAULT.
OCI_CONFIG_FILE = Path(os.getenv("OCI_CONFIG_FILE", "~/.oci/config")).expanduser()
OCI_PROFILE = os.getenv("OCI_PROFILE", "DEFAULT")
OCI_COMPARTMENT_ID = os.getenv("COMPARTMENT_OCID", COMPARTMENT_OCID)
OCI_REGION = os.getenv("OCI_REGION", os.getenv("AI_ENDPOINT_REGION"))
OCI_GENAI_CHAT_MODEL = os.getenv("OCI_GENAI_CHAT_MODEL", "xai.grok-4.3")
OCI_GENAI_EMBED_MODEL = os.getenv("OCI_GENAI_EMBED_MODEL", "cohere.embed-v4.0")
OCI_GENAI_EMBED_DIMENSIONS = os.getenv("OCI_GENAI_EMBED_DIMENSIONS", "512")
OCI_GENAI_EMBED_DIMENSIONS = int(OCI_GENAI_EMBED_DIMENSIONS) if OCI_GENAI_EMBED_DIMENSIONS else None
OCI_GENAI_ENDPOINT = os.getenv("ENDPOINT")
DEMO_NARRATIVE_COUNT = int(os.getenv("DEMO_NARRATIVE_COUNT", "4"))
OCI_EMBED_BATCH_SIZE = int(os.getenv("OCI_EMBED_BATCH_SIZE", "32"))
COPILOT_MAX_RELEVANT_RESULTS = int(os.getenv("COPILOT_MAX_RELEVANT_RESULTS", "8"))
COPILOT_MAX_RECENT_MESSAGES = int(os.getenv("COPILOT_MAX_RECENT_MESSAGES", "10"))
COPILOT_MAX_TOKENS = int(os.getenv("COPILOT_MAX_TOKENS", "700"))

_missing = []
if not OCI_CONFIG_FILE.exists():
    _missing.append(f"OCI_CONFIG_FILE not found: {OCI_CONFIG_FILE}")
if not OCI_COMPARTMENT_ID:
    _missing.append("OCI_COMPARTMENT_ID is not set")
if _missing:
    raise RuntimeError("OCI Generative AI configuration is incomplete:\n- " + "\n- ".join(_missing))

import oci
from oci.generative_ai import GenerativeAiClient
from oci.generative_ai_inference import GenerativeAiInferenceClient

oci_config = oci.config.from_file(str(OCI_CONFIG_FILE), profile_name=OCI_PROFILE)
OCI_REGION = OCI_REGION or oci_config.get("region")
OCI_GENAI_ENDPOINT = OCI_GENAI_ENDPOINT or f"https://inference.generativeai.{OCI_REGION}.oci.oraclecloud.com"

genai_client = GenerativeAiInferenceClient(
    config=oci_config,
    service_endpoint=OCI_GENAI_ENDPOINT,
)
genai_management_client = GenerativeAiClient(config=oci_config)


def list_oci_genai_models(capability: str = "CHAT", limit: int = 100) -> list[dict]:
    """List active OCI GenAI models visible in this compartment/region."""
    kwargs = {"lifecycle_state": "ACTIVE", "limit": limit}
    if capability:
        kwargs["capability"] = [capability]
    response = genai_management_client.list_models(OCI_COMPARTMENT_ID, **kwargs)
    items = getattr(response.data, "items", response.data)
    rows = []
    for item in items:
        data = oci.util.to_dict(item)
        rows.append({
            "id": data.get("id"),
            "display_name": data.get("display_name") or data.get("displayName"),
            "vendor": data.get("vendor"),
            "version": data.get("version"),
            "capabilities": data.get("capabilities") or data.get("capability"),
        })
    for row in rows:
        print(f"{row['id']}  |  {row['display_name']}  |  {row['vendor']}  |  {row['capabilities']}")
    return rows


print(f"Config ready: db={DB_USER}@{DB_DSN}")
print(f"OCI GenAI: region={OCI_REGION} profile={OCI_PROFILE} endpoint={OCI_GENAI_ENDPOINT}")
print(f"OCI models: chat={OCI_GENAI_CHAT_MODEL} | embed={OCI_GENAI_EMBED_MODEL}")
print("If the chat model is not found, run list_oci_genai_models('CHAT') to see valid model IDs for this region/compartment.")

import oracledb

# Oracle 26ai's native JSON datatype round-trips as Python dict/list via
# python-oracledb. fetch_lobs=False keeps CLOB columns coming back as str
# instead of LOB locators so the rest of the notebook stays simple.
oracledb.defaults.fetch_lobs = False


def connect_to_oracle():
    """Connect to Oracle AI Database using the DB_* config from Part 1."""
    return oracledb.connect(user=DB_USER, password=DB_PASSWORD, dsn=DB_DSN)

vector_conn = connect_to_oracle()
ok(f"Connected - using user {vector_conn.username}")



 Connected. Next: wire up OCI Generative AI embeddings + `oracleagentmemory`.

>  **Key insight - Part 1:** Oracle AI Database is a converged engine. The SDK's tables, our hand-rolled `CITY_ASSET` / `CITY_INSPECTION_FINDING` tables, and any other SQL all live on this same connection. Vector search via `VECTOR_DISTANCE()` works over OCI-generated vectors stored in Oracle's native `VECTOR` columns.


###  Optional: Reset the workspace

If you've run this workshop before, the SDK's tables (`CITY_THREAD`, `CITY_MESSAGE`, `CITY_MEMORY`, `CITY_RECORD_CHUNKS`, `CITY_ACTOR_PROFILE`) and the workshop's custom tables (`CITY_ASSET`, `CITY_INSPECTION_FINDING`) may still hold state from prior runs.


**Run the cell below to wipe everything CITY_*, then re-run from Part 2.** If you've already initialised the SDK (`memory = OracleAgentMemory(...)`) in Part 2, you'll need to **restart the kernel** after this reset - the in-memory `memory` and `thread` objects still reference the dropped tables.

Skip this cell on the first-ever run.


In [ ]:
# Optional: wipe all CITY_* tables for a clean slate.
# Safe to run multiple times; safe to skip.
_to_drop = (
    # Drop the hand-rolled tables first (they have FKs into CITY_ASSET).
    "CITY_INSPECTION_FINDING", "CITY_ASSET",
    # Then the SDK's tables (FK-cascaded in this order).
    "CITY_MEMORY", "CITY_MESSAGE", "CITY_RECORD_CHUNKS",
    "CITY_THREAD", "CITY_ACTOR_PROFILE",
    "CITY_ORACLEAGENTMEMORY_SCHEMA_META",
)
with vector_conn.cursor() as cur:
    for t in _to_drop:
        try:
            cur.execute(f'DROP TABLE "{t}" CASCADE CONSTRAINTS')
            print(f"  dropped {t}")
        except Exception:
            pass  # table didn't exist - fine
vector_conn.commit()
ok("Workspace clean. RESTART THE KERNEL before re-running Part 2 if you've already done it.")


# Part 2: The embedder and SDK


The SDK needs an embedder to turn text into vectors. This adapter exposes **OCI Generative AI embeddings** through the SDK's `IEmbedder` interface, so the same `cohere.embed-v4.0` provider powers both `oracleagentmemory` semantic search and the custom `CITY_INSPECTION_FINDING` vector search.


In [ ]:
from oracleagentmemory.apis.embedders.embedder import IEmbedder
from oci.generative_ai_inference.models import EmbedTextDetails, OnDemandServingMode
import numpy as np
import asyncio


class OCIGenAIEmbedder(IEmbedder):
    """Bridges OCI Generative AI embeddings to the SDK's IEmbedder."""

    def __init__(
        self,
        client: GenerativeAiInferenceClient,
        compartment_id: str = OCI_COMPARTMENT_ID,
        model_id: str = OCI_GENAI_EMBED_MODEL,
        output_dimensions: int | None = OCI_GENAI_EMBED_DIMENSIONS,
        debug: bool = False,
    ):
        self.client = client
        self.compartment_id = compartment_id
        self.model_id = model_id
        self.output_dimensions = output_dimensions
        self.debug = debug
        self.serving_mode = OnDemandServingMode(model_id=model_id)

    def embed(self, texts: list[str], *, is_query: bool = False) -> np.ndarray:
        if not texts:
            cols = self.output_dimensions or 0
            return np.empty((0, cols), dtype=np.float32)

        kwargs = {
            "inputs": list(texts),
            "serving_mode": self.serving_mode,
            "compartment_id": self.compartment_id,
            "input_type": "SEARCH_QUERY" if is_query else "SEARCH_DOCUMENT",
            "truncate": "END",
            "embedding_types": ["float"],
        }
        if self.output_dimensions:
            kwargs["output_dimensions"] = self.output_dimensions
        details = EmbedTextDetails(**kwargs)
        response = self.client.embed_text(details)

        def candidates(obj):
            if obj is None:
                return []
            values = [obj]
            if isinstance(obj, dict):
                for key in (
                    "float", "FLOAT", "embeddings", "embeddingsByType",
                    "embeddings_by_type", "values", "data", "embedContents",
                    "embed_contents", "embedding", "vector",
                ):
                    if key in obj:
                        values.extend(candidates(obj[key]))
            elif isinstance(obj, (list, tuple)):
                for item in obj:
                    values.extend(candidates(item))
            else:
                try:
                    as_dict = oci.util.to_dict(obj)
                except Exception:
                    as_dict = None
                if as_dict:
                    values.extend(candidates(as_dict))
                for attr in (
                    "float", "FLOAT", "float_", "embeddings", "values", "data",
                    "embeddings_by_type", "embeddingsByType", "embed_contents",
                    "embedContents", "embedding", "vector",
                ):
                    try:
                        attr_value = getattr(obj, attr)
                    except Exception:
                        continue
                    values.extend(candidates(attr_value))
            return values

        def to_array(vectors):
            if vectors is None:
                return None
            try:
                arr = np.asarray(vectors, dtype=np.float32)
            except (TypeError, ValueError):
                return None
            if arr.ndim == 0:
                return None
            if arr.ndim == 1 and len(texts) == 1:
                arr = arr.reshape(1, -1)
            return arr if arr.ndim == 2 else None

        arr = None
        checked = []
        for value in candidates(response.data):
            raw_shape = getattr(np.asarray(value, dtype=object), "shape", None)
            candidate = to_array(value)
            checked.append(f"{type(value).__name__}:raw={raw_shape}:usable={getattr(candidate, 'shape', None)}")
            if candidate is not None:
                arr = candidate
                break
        if self.debug:
            print("OCI embedding response type:", type(response.data).__name__)
            print("OCI embedding response fields:", response.data.attribute_map)
            print("OCI embedding candidates:", checked)
        if arr is None:
            raise RuntimeError(
                "OCI embedding response did not include a 2-D float embedding array. "
                f"Checked candidates: {checked}; "
                f"available response fields: {response.data.attribute_map}"
            )
        return arr

    async def embed_async(self, texts: list[str], *, is_query: bool = False) -> np.ndarray:
        return await asyncio.to_thread(self.embed, texts, is_query=is_query)


# Verify the OCI embedder
embedder = OCIGenAIEmbedder(genai_client, debug=True)
_v = embedder.embed(["corrosion on bearing assembly"])
if _v.ndim != 2 or _v.shape[0] != 1:
    raise RuntimeError(
        f"Expected one 2-D embedding row, got shape {_v.shape}. "
        "Restart the kernel and rerun this whole cell so the latest OCIGenAIEmbedder class is active."
    )
OCI_EMBED_DIM = int(_v.shape[-1])
if OCI_GENAI_EMBED_DIMENSIONS:
    assert OCI_EMBED_DIM == OCI_GENAI_EMBED_DIMENSIONS, f"Expected {OCI_GENAI_EMBED_DIMENSIONS} dims, got {OCI_EMBED_DIM}"
assert _v.dtype == np.float32, f"Expected float32, got {_v.dtype}"
ok(f"Embedder ready - OCI embedder returns {_v.dtype} ({_v.shape}) arrays")


### SDK initialization (pre-built)

Wires up `OracleAgentMemory` with OCI Generative AI for its internal extractor LLM and the OCI-backed embedder from the previous cell. No OCI Generative AI server or in-database OCI GenAI model is required.

- `extract_memories=True` turns on automatic LLM extraction of `fact` / `preference` / `guideline` / `memory` on every `add_messages` call.
- `table_name_prefix="CITY_"` namespaces the SDK's tables.


In [ ]:
from oracleagentmemory.core import OracleAgentMemory, SchemaPolicy
from oracleagentmemory.core.llms.llm import Llm
from oracleagentmemory.apis.thread import Message
from oci.generative_ai_inference.models import (
    AssistantMessage,
    ChatDetails,
    GenericChatRequest,
    OnDemandServingMode,
    SystemMessage,
    TextContent,
    UserMessage,
)
from types import SimpleNamespace


def _oci_text_content(text: str) -> list:
    return [TextContent(text=text)]


def _message_role_content(message) -> tuple[str, str]:
    if isinstance(message, str):
        return "user", message
    if isinstance(message, dict):
        role = message.get("role") or "user"
        content = message.get("content", "")
    else:
        role = getattr(message, "role", "user")
        content = getattr(message, "content", str(message))
    if isinstance(content, list):
        content = "".join(
            item.get("text", "") if isinstance(item, dict) else getattr(item, "text", str(item))
            for item in content
        )
    return str(role).lower(), str(content)


def _coerce_messages(messages) -> list:
    if messages is None:
        return []
    if isinstance(messages, str):
        return [{"role": "user", "content": messages}]
    return list(messages)


def _to_oci_message(message):
    role, content = _message_role_content(message)
    if role == "system":
        return SystemMessage(content=_oci_text_content(content))
    if role == "assistant":
        return AssistantMessage(content=_oci_text_content(content))
    return UserMessage(content=_oci_text_content(content))


def _extract_oci_chat_text(response) -> str:
    chat_response = response.data.chat_response
    choice = chat_response.choices[0]
    content = choice.message.content or []
    parts = []
    for item in content:
        text = getattr(item, "text", None)
        if text:
            parts.append(text)
    return "".join(parts)


def call_oci_genai_chat_sync(messages, *, max_tokens: int = 700, temperature: float = 0) -> SimpleNamespace:
    """Call OCI Generative AI through the native OCI SDK."""
    chat_request = GenericChatRequest(
        messages=[_to_oci_message(m) for m in _coerce_messages(messages)],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    details = ChatDetails(
        compartment_id=OCI_COMPARTMENT_ID,
        serving_mode=OnDemandServingMode(model_id=OCI_GENAI_CHAT_MODEL),
        chat_request=chat_request,
    )
    try:
        response = genai_client.chat(details)
    except oci.exceptions.ServiceError as exc:
        if exc.status == 404:
            raise RuntimeError(
                f"OCI GenAI chat model {OCI_GENAI_CHAT_MODEL!r} was not found in "
                f"region {OCI_REGION!r} using endpoint {OCI_GENAI_ENDPOINT!r}. "
                "Run list_oci_genai_models('CHAT') and set OCI_GENAI_CHAT_MODEL "
                "to one of the returned IDs, or switch OCI_REGION/OCI_GENAI_ENDPOINT "
                "to a region where that model is available."
            ) from exc
        raise
    content = _extract_oci_chat_text(response)
    return SimpleNamespace(choices=[
        SimpleNamespace(message=SimpleNamespace(content=content))
    ])


async def call_oci_genai_chat(messages: list, model: str = None, max_tokens: int = 700):
    """Async wrapper used by the notebook copilot cells."""
    return await asyncio.to_thread(call_oci_genai_chat_sync, messages, max_tokens=max_tokens)


def _extract_json_object(text: str) -> str:
    """Return the first top-level JSON object in `text`, stripping any markdown
    code fences or surrounding prose. The SDK's extractor validates this output
    with Pydantic, so it must be clean JSON."""
    import re
    if not text:
        return text
    m = re.search(r"```(?:json)?(.*?)```", text, re.S)
    if m:
        text = m.group(1)
    start, end = text.find("{"), text.rfind("}")
    return text[start:end + 1].strip() if (start != -1 and end > start) else text.strip()


def _coerce_extraction_json(text: str) -> str:
    """Make the extractor LLM's JSON satisfy the SDK's strict schema. The OCI
    model emits only {text, record_type} per memory, but the SDK's _Memory model
    requires id/scope/entities/timestamp/valid_to/importance to be present (most
    are nullable) plus a top-level `thoughts`. Without filling these, the SDK's
    model_validate_json fails and 0 memories get stored."""
    import json, uuid
    cleaned = _extract_json_object(text)
    try:
        data = json.loads(cleaned)
    except (ValueError, TypeError):
        return cleaned  # let the SDK surface a genuine parse error
    if not isinstance(data, dict):
        return cleaned
    data.setdefault("thoughts", None)
    valid_types = {"memory", "guideline", "fact", "preference"}
    coerced = []
    for mem in (data.get("memories") or []):
        if not isinstance(mem, dict):
            continue
        txt = mem.get("text")
        if not (isinstance(txt, str) and txt.strip()):
            continue  # drop memories with no usable text
        if mem.get("record_type") not in valid_types:
            mem["record_type"] = "memory"
        mem.setdefault("id", uuid.uuid4().hex[:16])
        mem.setdefault("scope", None)
        if not isinstance(mem.get("entities"), list):
            mem["entities"] = []
        mem.setdefault("timestamp", None)
        mem.setdefault("valid_to", None)
        mem.setdefault("importance", None)
        coerced.append(mem)
    data["memories"] = coerced
    return json.dumps(data)


class OCIGenAIOracleMemoryLlm(Llm):
    """Duck-typed SDK LLM adapter backed by OCI Generative AI chat.

    The notebook keeps `oracleagentmemory` as the memory system. This adapter
    uses OCI GenAI for the SDK extractor's model calls.
    Multiple method aliases are provided because the SDK only needs one of them,
    depending on its internal call path.
    """

    def __init__(self, model: str = OCI_GENAI_CHAT_MODEL):
        self.model = model

    def _complete(self, messages=None, **kwargs):
        messages = messages or kwargs.get("messages") or kwargs.get("prompt") or kwargs.get("input")
        return call_oci_genai_chat_sync(
            messages,
            max_tokens=kwargs.get("max_tokens", 700),
            temperature=kwargs.get("temperature", 0),
        )

    async def _acomplete(self, messages=None, **kwargs):
        return await asyncio.to_thread(self._complete, messages, **kwargs)

    def completion(self, messages=None, **kwargs):
        return self._complete(messages, **kwargs)

    async def acompletion(self, messages=None, **kwargs):
        return await self._acomplete(messages, **kwargs)

    def chat_completion(self, messages=None, **kwargs):
        return self._complete(messages, **kwargs)

    async def achat_completion(self, messages=None, **kwargs):
        return await self._acomplete(messages, **kwargs)

    def generate(self, prompt=None, response_json_schema=None, **kwargs):
        # The SDK's extractor validates this output with Pydantic
        # (_MemoryExtractionPayload.model_validate_json), so it must be clean,
        # untruncated JSON. The default 700-token cap truncates multi-record
        # extraction, and OCI/Llama often wraps JSON in ```json fences or prose -
        # both make validation fail silently (0 memories). Give it room and
        # strip the output down to the JSON object. The copilot path
        # (response_json_schema is None) is left untouched.
        response = self._complete(
            prompt,
            max_tokens=kwargs.get("max_tokens", 2048 if response_json_schema is not None else 700),
            temperature=kwargs.get("temperature", 0),
        )
        text = response.choices[0].message.content or ""
        if response_json_schema is not None:
            text = _coerce_extraction_json(text)
        return SimpleNamespace(text=text, raw=response)

    async def generate_async(self, prompt=None, response_json_schema=None, **kwargs):
        return await asyncio.to_thread(
            self.generate,
            prompt,
            response_json_schema=response_json_schema,
            **kwargs,
        )

    async def agenerate(self, messages=None, **kwargs):
        return await self.generate_async(messages, **kwargs)

    def invoke(self, messages=None, **kwargs):
        return self._complete(messages, **kwargs)

    async def ainvoke(self, messages=None, **kwargs):
        return await self._acomplete(messages, **kwargs)

    def __call__(self, messages=None, **kwargs):
        return self._complete(messages, **kwargs)


sdk_llm = OCIGenAIOracleMemoryLlm()

memory = OracleAgentMemory(
    connection=vector_conn,
    embedder=embedder,
    llm=sdk_llm,
    extract_memories=True,
    schema_policy=SchemaPolicy.CREATE_IF_NECESSARY,
    table_name_prefix="CITY_",
)
ok("OracleAgentMemory ready. Tables under prefix CITY_* using OCI GenAI models")


# Part 3: City asset + auto-extraction

> **Component 1 of 3 · build → verify.** You'll build `report_event` and watch the SDK auto-extract typed memories from raw narratives. You verify it in isolation here; the copilot wires it in at Part 6 - as the context card and the write-back step.


Two pieces in this part:

1. The **`CITY_ASSET` table** - 26 real urban infrastructure assets, loaded from `data/maintenance_logs.json` + `data/inspection_reports.json`. Hand-rolled SQL table outside the SDK.
2. **Auto-extraction from real maintenance narratives** - the SDK's extractor turns paragraph-long inspection write-ups into typed `fact` / `preference` / `guideline` records inside `CITY_MEMORY`.

Both pre-built cells run automatically; the TODOs are the auto-extraction logic.


In [ ]:
# Pre-built: load the real asset registry from the committed JSON files.
# The 26 assets are the union of asset_name fields across both datasets.
import json
from pathlib import Path

_data_dir = Path("data") if Path("data").exists() else Path("../data")
with open(_data_dir / "maintenance_logs.json") as f:
    _logs = json.load(f)
with open(_data_dir / "inspection_reports.json") as f:
    _reports = json.load(f)

_asset_names = sorted(set(x["asset_name"] for x in _logs) | set(x["asset_name"] for x in _reports))
print(f"Loaded {len(_logs)} maintenance logs, {len(_reports)} inspection reports.")
print(f"Discovered {len(_asset_names)} unique assets.")

# Pre-built: classify each asset by name heuristics into one of 8 asset classes.
def classify_asset(name: str) -> str:
    n = name.lower()
    if "bridge" in n or "overpass" in n: return "bridge"
    if "substation" in n:                  return "substation"
    if "pipeline" in n:                    return "pipeline"
    if "water" in n or "outfall" in n or "treatment" in n: return "water"
    if "solar" in n or "gas distribution" in n:           return "energy"
    if "sensor" in n or "array" in n or "monitor" in n or "gauge" in n: return "sensor"
    if "tower" in n or "relay" in n:      return "comms"
    if "seawall" in n or "retaining" in n or "booster" in n: return "civil"
    return "other"

equipment = [{"asset_id": name, "asset_class": classify_asset(name)} for name in _asset_names]
from collections import Counter
print("Asset class distribution:", dict(Counter(e['asset_class'] for e in equipment)))
print("\nFirst 5 assets:")
for e in equipment[:5]:
    print(f"  {e['asset_id']:40} -> {e['asset_class']}")

# Pre-built: create PLANT_ASSET... wait, no - CITY_ASSET - and bulk-INSERT all 26.
with vector_conn.cursor() as cur:
    try:
        cur.execute("DROP TABLE CITY_INSPECTION_FINDING CASCADE CONSTRAINTS")
    except Exception:
        pass
    try:
        cur.execute("DROP TABLE CITY_ASSET CASCADE CONSTRAINTS")
    except Exception:
        pass
    cur.execute("""
        CREATE TABLE CITY_ASSET (
            asset_id     VARCHAR2(128) PRIMARY KEY,
            asset_class  VARCHAR2(32) NOT NULL,
            created_at   TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    cur.executemany(
        "INSERT INTO CITY_ASSET (asset_id, asset_class) VALUES (:1, :2)",
        [(e["asset_id"], e["asset_class"]) for e in equipment],
    )
vector_conn.commit()
print(f"Inserted {len(equipment)} rows into CITY_ASSET.")

# Helper: look up one asset row as a dict.
def get_asset(asset_id: str) -> dict | None:
    with vector_conn.cursor() as cur:
        cur.execute(
            "SELECT asset_id, asset_class FROM CITY_ASSET WHERE asset_id = :id",
            id=asset_id,
        )
        row = cur.fetchone()
    if not row:
        return None
    return {"asset_id": row[0], "asset_class": row[1]}

print(get_asset("Harbor Bridge"))
ok("Finished running cell.")


### Auto-extraction

When you call `thread.add_messages(...)` with `extract_memories=True`, the SDK runs its extractor LLM internally. It reads the new message + cached running summary + past memories, and outputs typed records (`fact` / `preference` / `guideline` / `memory`) into `CITY_MEMORY`.

![The SDK extractor: add_messages forks to CITY_MESSAGE and the extractor LLM, which distills typed records into CITY_MEMORY](diagrams/extractor.svg)

**You don't write the extraction prompt.** The SDK does.


The get-or-create thread logic is pre-built. **You write the two lines that trigger extraction:** build a `content` string that names both the asset and the inspector (the extractor only sees text, not metadata) - `f"[Asset: {asset_id}] [Inspector: {inspector}] {narrative}"` - then return `await t.add_messages_async([Message(role="user", content=content)])`.


In [ ]:

async def report_event(asset_id: str, inspector: str, narrative: str, thread_id: str) -> list:
    """Persist a maintenance event narrative and trigger SDK auto-extraction."""
    try:
        t = memory.get_thread(thread_id)
    except Exception:
        t = None
    if t is None:
        t = memory.create_thread(
            user_id=inspector,
            thread_id=thread_id,
            agent_id="CITY",
            # Keep a rolling <=2-sentence thread summary, refreshed every 2 messages.
            enable_context_summary=True,
            context_summary_update_frequency=2,
        )
    # TODO 1: build the `content` string, then return the result of
    # await t.add_messages_async([...]). The content must name BOTH the asset
    # and the inspector (the extractor only sees text, not metadata), e.g.
    # f"[Asset: {asset_id}] [Inspector: {inspector}] {narrative}".
    # YOUR CODE HERE (2 lines)
    pass

# Checkpoint: TODO 1 - smoke test with one real narrative from the dataset
_seed = _logs[0]  # First maintenance log - likely Harbor Bridge routine inspection
with timed("TODO 1 report_event (SDK write + LLM extraction)"):
    ids = await report_event(
        asset_id=_seed["asset_name"],
        inspector="smoke_inspector",
        narrative=_seed["narrative"],
        thread_id="smoke_thread",
    )
assert ids and len(ids) >= 1, "TODO 1 incomplete - add_messages should return at least one ID"
ok(f"TODO 1 passed - added message(s): {ids}")


<details>
<summary>💡 Show answer (TODO 1)</summary>

```python
content = f"[Asset: {asset_id}] [Inspector: {inspector}] {narrative}"
return await t.add_messages_async([Message(role="user", content=content)])
```

</details>

### See what got persisted

When `report_event` ran, two things happened inside the SDK:

1. **One row INSERTed into `CITY_MESSAGE`** - the raw narrative
2. **The extractor LLM ran** and INSERTed 0+ typed records into `CITY_MEMORY`

The helper below shows both tables. Run it now to see the smoke-test state, then again after TODO 2.


In [ ]:
def peek_sdk_tables(thread_id: str = None) -> None:
    """Print rows from CITY_MESSAGE and CITY_MEMORY (optionally scoped to one thread)."""
    where = "WHERE thread_id = :tid" if thread_id else ""
    bind = {"tid": thread_id} if thread_id else {}
    with vector_conn.cursor() as cur:
        cur.execute(f"""
            SELECT message_role, SUBSTR(content, 1, 100) AS preview, user_id, agent_id
              FROM CITY_MESSAGE {where}
             ORDER BY order_seq
        """, bind)
        msgs = cur.fetchall()
        print(f" CITY_MESSAGE - {len(msgs)} row(s)" + (f" for thread={thread_id}" if thread_id else ""))
        for role, preview, uid, aid in msgs:
            text = preview.read() if hasattr(preview, 'read') else preview
            print(f"   [{role:9}] user={uid or '-':18} agent={aid or '-':6} | {text}")

        cur.execute(f"""
            SELECT memory_type, SUBSTR(content, 1, 100) AS preview, user_id, agent_id
              FROM CITY_MEMORY {where}
             ORDER BY order_seq
        """, bind)
        mems = cur.fetchall()
        print(f"\n CITY_MEMORY - {len(mems)} row(s)" + (f" for thread={thread_id}" if thread_id else ""))
        for mtype, preview, uid, aid in mems:
            text = preview.read() if hasattr(preview, 'read') else preview
            print(f"   [{mtype:10}] user={uid or '-':18} agent={aid or '-':6} | {text}")

peek_sdk_tables(thread_id="smoke_thread")

ok("Cell is complete")


Scope-matching on `memory.search` is a feature, not a quirk: records inherit `user_id` from the thread that wrote them, so a search has to ask for that same `user_id` to see them. The loop is pre-built. **Set `user_id` on the search to `"inspector_demo"`** - the string the loop wrote with. Leave it as the default `None` and the SDK matches `user_id IS NULL`, so you'll get nothing back.


In [ ]:

# Pre-built helper: stratified sample of narratives.
def sample_narratives(n: int = 4) -> list:
    """Stratified sample: mix of severities, multiple assets."""
    import random
    random.seed(42)  # Reproducible across re-runs
    buckets = {"routine": [], "warning": [], "critical": []}
    for log in _logs:
        buckets.get(log["severity"], []).append(log)
    # Take roughly proportional sample
    n_routine, n_warning, n_critical = max(1, n // 2), max(1, n // 3), max(1, n - n // 2 - n // 3)
    picks = (random.sample(buckets["routine"], min(n_routine, len(buckets["routine"]))) +
             random.sample(buckets["warning"], min(n_warning, len(buckets["warning"]))) +
             random.sample(buckets["critical"], min(n_critical, len(buckets["critical"]))))
    return picks[:n]

with timed(f"sample_narratives({DEMO_NARRATIVE_COUNT})"):
    narratives = sample_narratives(DEMO_NARRATIVE_COUNT)
print("Sampled narratives (asset, severity):")
for n in narratives:
    print(f"  {n['asset_name']:40} [{n['severity']}]")

for idx, narr in enumerate(narratives, start=1):
    with timed(f"report_event {idx}/{len(narratives)} (SDK write + extraction)"):
        await report_event(
            asset_id=narr["asset_name"],
            inspector="inspector_demo",
            narrative=narr["narrative"],
            thread_id="inspect_demo",
        )

# The SDK's high-level search requires a specific user_id (rejects
# exact_user_match=False). The records inherit user_id from the thread,
# which we created with inspector="inspector_demo".
with timed("memory.search_async (embed query + vector search)"):
    results = await memory.search_async(
        query="recurring asset concerns and inspector practices",
        user_id=None,  # TODO 2: set to the inspector the loop wrote with ("inspector_demo")
        agent_id="CITY",
        record_types=["fact", "preference", "guideline", "memory"],
        max_results=30,
    )
for r in results:
    print(f"  [{r.record.record_type:11s}] {r.record.content}")

ok("Cell is complete")


<details>
<summary>💡 Show answer (TODO 2)</summary>

```python
user_id="inspector_demo",
```

</details>

**Checkpoint for TODO 2.** Confirms the 4 narratives were extracted into searchable memories, and that a scope-matched search (same `user_id`) returns them.


In [ ]:
# Reuse the records the TODO 2 search returned above - no need to search again.
assert len(results) >= 3, (
    f"TODO 2 - expected at least 3 extracted records from {DEMO_NARRATIVE_COUNT} narratives, got {len(results)}.\n"
    "Check that OCI Generative AI credentials/models are reachable and that you called report_event for all narratives."
)
ok(f"TODO 2 passed - {len(results)} memories extracted from {DEMO_NARRATIVE_COUNT} narratives")

### Peek at the tables again - After 4 narratives

Same helper, this time scoped to the `inspect_demo` thread. You should see 4 user-role messages in `CITY_MESSAGE` and several typed records in `CITY_MEMORY`. Notice that none of the records are custom types - the SDK rejects them. All extracted records are one of the four natives.


In [ ]:
peek_sdk_tables(thread_id="inspect_demo")

ok("Cell is complete")


# Part 4: Inspection findings + similar-finding search

> **Component 2 of 3 · build → verify.** You'll build `log_finding` and `find_similar_findings`, then test the vector search on its own. The copilot calls it at Part 6 to surface similar past findings on the same asset.


An **inspection finding** isn't a `fact` or a `preference` - it's a structured domain object with `category`, `severity`, `description`, `recommendation`, `inspector`, and `overall_grade`. The SDK rejects custom `record_type` values, so findings live in their own hand-rolled `CITY_INSPECTION_FINDING` SQL table with a `VECTOR(OCI_EMBED_DIM)` column + HNSW index.


In [ ]:
# Pre-built: create CITY_INSPECTION_FINDING with OCI embedding dimension + HNSW index.
with vector_conn.cursor() as cur:
    cur.execute(f"""
        CREATE TABLE CITY_INSPECTION_FINDING (
            finding_id      VARCHAR2(64) PRIMARY KEY,
            asset_id        VARCHAR2(128) NOT NULL,
            inspector       VARCHAR2(128),
            overall_grade   VARCHAR2(2),
            category        VARCHAR2(32),
            severity        VARCHAR2(16),
            description     CLOB NOT NULL,
            recommendation  CLOB,
            days_ago        NUMBER,
            embedding       VECTOR({OCI_EMBED_DIM}) NOT NULL,
            created_at      TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            CONSTRAINT fk_finding_asset FOREIGN KEY (asset_id) REFERENCES CITY_ASSET(asset_id)
        )
    """)
    try:
        cur.execute("""
            CREATE VECTOR INDEX city_finding_embedding_idx
            ON CITY_INSPECTION_FINDING (embedding)
            ORGANIZATION INMEMORY NEIGHBOR GRAPH
            DISTANCE COSINE
            WITH TARGET ACCURACY 95
            PARAMETERS (TYPE HNSW, M 16, EFCONSTRUCTION 200)
        """)
    except Exception as e:
        print(f"  (skipped HNSW index: {e})")
vector_conn.commit()
ok(f"CITY_INSPECTION_FINDING created with VECTOR({OCI_EMBED_DIM}) embedding column.")


With the table created, load the real data: embed each of the ~220 inspection findings with OCI Generative AI and bulk-insert them. They're already structured, so no LLM extraction is needed - just embed the description and `INSERT`.


In [ ]:

# Pre-built: bulk-load all ~220 findings from the inspection reports.
# No LLM extraction needed - findings are already structured. Just OCI-embed + INSERT.
import array, uuid

finding_items = []
for report in _reports:
    for finding in report["findings"]:
        finding_items.append((report, finding))

descriptions = [finding["description"] for _, finding in finding_items]
vectors = []
with timed(f"OCI embed {len(descriptions)} finding descriptions (batch={OCI_EMBED_BATCH_SIZE})"):
    for start in range(0, len(descriptions), OCI_EMBED_BATCH_SIZE):
        batch = descriptions[start:start + OCI_EMBED_BATCH_SIZE]
        vectors.extend(embedder.embed(batch))

rows = []
for (report, finding), vec in zip(finding_items, vectors):
    rows.append({
        "finding_id":     str(uuid.uuid4())[:12],
        "asset_id":       report["asset_name"],
        "inspector":      report["inspector"],
        "overall_grade":  report["overall_grade"],
        "category":       finding["category"],
        "severity":       finding["severity"],
        "description":    finding["description"],
        "recommendation": finding["recommendation"],
        "days_ago":       report["days_ago"],
        "embedding":      array.array('f', vec.tolist()),
    })

with timed(f"insert {len(rows)} rows into CITY_INSPECTION_FINDING"):
    with vector_conn.cursor() as cur:
        cur.executemany("""
            INSERT INTO CITY_INSPECTION_FINDING
              (finding_id, asset_id, inspector, overall_grade, category, severity,
               description, recommendation, days_ago, embedding)
            VALUES (:finding_id, :asset_id, :inspector, :overall_grade, :category, :severity,
                    :description, :recommendation, :days_ago, :embedding)
        """, rows)
    vector_conn.commit()
ok(f"Inserted {len(rows)} findings into CITY_INSPECTION_FINDING.")


`log_finding` is how a *new* finding gets recorded going forward - it embeds the description with OCI Generative AI and inserts one structured row into `CITY_INSPECTION_FINDING`. This is the deliberate write path the copilot itself never takes.


In [ ]:
import array, uuid

def log_finding(
    asset_id: str,
    inspector: str,
    category: str,
    severity: str,
    description: str,
    recommendation: str = "",
    overall_grade: str = None,
    days_ago: int = 0,
) -> str:
    """Persist a new inspection finding into CITY_INSPECTION_FINDING."""
    finding_id = str(uuid.uuid4())[:12]
    vec = array.array('f', embedder.embed([description])[0].tolist())
    with vector_conn.cursor() as cur:
        cur.execute(
            """INSERT INTO CITY_INSPECTION_FINDING
                 (finding_id, asset_id, inspector, overall_grade, category, severity,
                  description, recommendation, days_ago, embedding)
               VALUES (:finding_id, :asset_id, :inspector, :overall_grade, :category, :severity,
                       :description, :recommendation, :days_ago, :embedding)""",
            finding_id=finding_id, asset_id=asset_id, inspector=inspector,
            overall_grade=overall_grade, category=category, severity=severity,
            description=description, recommendation=recommendation,
            days_ago=days_ago, embedding=vec,
        )
    vector_conn.commit()
    return finding_id

ok("Cell is complete")


**Verify `log_finding`.** Logs a test finding and confirms it lands in `CITY_INSPECTION_FINDING`.


In [ ]:
_fid = log_finding(
    asset_id="Harbor Bridge",
    inspector="checkpoint_test",
    category="corrosion",
    severity="medium",
    description="Surface corrosion on pier 2 bearing assemblies; ~25% section loss observed.",
    recommendation="Remove corrosion, apply primer + finish coat within 60 days.",
    overall_grade="C",
    days_ago=0,
)
assert _fid and isinstance(_fid, str), "log_finding should return a finding_id string"

with vector_conn.cursor() as cur:
    cur.execute(
        "SELECT COUNT(*) FROM CITY_INSPECTION_FINDING WHERE inspector = :i",
        i="checkpoint_test",
    )
    n = cur.fetchone()[0]
assert n == 1, "log_finding test finding not retrievable"
ok(f"log_finding works - finding_id={_fid}")


This is the converged-DB pattern: vector similarity and relational filters in one statement. The embedding wrap, cursor execution, and CLOB handling are pre-built - **you write the SQL:**

```sql
SELECT finding_id, asset_id, inspector, overall_grade, category, severity,
       description, recommendation, days_ago,
       VECTOR_DISTANCE(embedding, :q, COSINE) AS score
  FROM CITY_INSPECTION_FINDING
 WHERE (:asset_id IS NULL OR asset_id = :asset_id)
   AND (:category IS NULL OR category = :category)
 ORDER BY score
 FETCH FIRST :k ROWS ONLY
```

The `(:bind IS NULL OR col = :bind)` trick lets one query do double duty - search every asset when `asset_id` is `None`, or narrow to one when it's set. Same for `category`.


In [ ]:
def find_similar_findings(description: str, asset_id: str = None, category: str = None, k: int = 3) -> list:
    """Vector-search CITY_INSPECTION_FINDING, optionally narrowed to one asset and/or category.

    Returns a list of dicts with all the structured fields + a `score` (cosine distance).
    """
    import array
    query_vec = array.array('f', embedder.embed([description], is_query=True)[0].tolist())
    # TODO 3: write the SQL. Select the finding columns plus
    # VECTOR_DISTANCE(embedding, :q, COSINE) AS score from CITY_INSPECTION_FINDING;
    # use the (:bind IS NULL OR col = :bind) trick to optionally filter by
    # asset_id and category; ORDER BY score; FETCH FIRST :k ROWS ONLY.
    sql = """
        -- YOUR SQL HERE
    """
    with vector_conn.cursor() as cur:
        cur.execute(sql, q=query_vec, asset_id=asset_id, category=category, k=k)
        cols = [d[0].lower() for d in cur.description]
        rows = []
        for r in cur.fetchall():
            row = dict(zip(cols, r))
            for key in ("description", "recommendation"):
                v = row.get(key)
                if v is not None and hasattr(v, "read"):
                    row[key] = v.read()
            rows.append(row)
    return rows

ok("Cell is complete")


<details>
<summary>💡 Show answer (TODO 3)</summary>

```sql
SELECT finding_id, asset_id, inspector, overall_grade, category, severity,
       description, recommendation, days_ago,
       VECTOR_DISTANCE(embedding, :q, COSINE) AS score
  FROM CITY_INSPECTION_FINDING
 WHERE (:asset_id IS NULL OR asset_id = :asset_id)
   AND (:category IS NULL OR category = :category)
 ORDER BY score
 FETCH FIRST :k ROWS ONLY
```

</details>

**Checkpoint for TODO 3.** Runs `find_similar_findings` and confirms the vector search returns relevant results, with the asset and category filters applied.


In [ ]:
_broad = find_similar_findings("bearing corrosion at piers", k=5)
_bridge = find_similar_findings("bearing corrosion at piers", asset_id="Harbor Bridge", k=5)
_corrosion_only = find_similar_findings("bearing corrosion at piers", category="corrosion", k=5)

assert _broad and len(_broad) >= 3, f"TODO 3 - broad search returned {len(_broad) if _broad else 0} hits"
assert _bridge and all(r["asset_id"] == "Harbor Bridge" for r in _bridge), \
    "TODO 3 - asset_id filter should restrict results to Harbor Bridge"
assert _corrosion_only and all(r["category"] == "corrosion" for r in _corrosion_only), \
    "TODO 3 - category filter should restrict results to corrosion only"
print(f"TODO 3 passed - broad={len(_broad)}, asset-filtered={len(_bridge)}, category-filtered={len(_corrosion_only)}")

for r in _bridge[:3]:
    print(f"  score={r['score']:.3f}  [{r['category']}/{r['severity']}]  {str(r['description'])[:90]}")

ok("Cell complete")


# Part 5: Scoping - Inspector vs City

> **Component 3 of 3 · build → verify.** You'll prove that one inspector can't read another's private notes - the multi-tenant safety guarantee the copilot depends on every turn.


Three scope dimensions: `user_id` (inspector), `agent_id` (`CITY`), `thread_id` (per asset). Enforced as SQL `WHERE` predicates - cross-user leakage is impossible at the DB layer.

**No TODO** - just run the demo cells to see scoping in action.


In [ ]:
# Inspector Mercer writes a personal note (user-scoped, invisible to others)
memory.add_memory(
    content="Remember to swap shifts with Jordan next Tuesday.",
    user_id="Evelyn_H_Mercer",
)

# Mercer also writes a city-wide tribal-knowledge guideline (agent-scoped)
memory.add_memory(
    content="On Harbor Bridge, inspect Pier 2 bearings annually - corrosion-prone since 2024.",
    agent_id="CITY",
)
print("Mercer wrote one personal memory and one city-wide memory.")

# Inspector Vance searches at user scope - should NOT see Mercer's personal note
vance_personal = await memory.search_async(
    query="shift swap notes",
    user_id="Jordan_Vance",
    record_types=["memory"],
    max_results=10,
)

# Vance searches at city scope - SHOULD see the Pier 2 guideline
vance_city = await memory.search_async(
    query="Harbor Bridge Pier 2 bearings",
    user_id=None,           # explicitly leave user dimension unconstrained
    agent_id="CITY",
    record_types=["memory"],
    max_results=10,
)

print(f"  Vance's personal-scope hits for 'shift swap': {len(vance_personal)}  (should be 0)")
print(f"  Vance's city-scope hits for 'Pier 2 bearings': {len(vance_city)}  (should be ≥ 1)")

# Assertions - multi-tenancy is enforced at the SQL layer
assert all("shift swap" not in r.record.content.lower() for r in vance_personal), \
    "Cross-inspector leak: Mercer's personal note showed up in Vance's user-scoped search"
assert any("Pier 2" in r.record.content for r in vance_city), \
    "Pier 2 guideline not retrievable at agent scope - check agent_id wiring"
ok("Multi-tenancy verified: Vance sees city-wide guidelines but NOT Mercer's personal notes.")


>  **Key insight - Part 5:** scoping is enforced as a SQL `WHERE` clause on `user_id` / `agent_id` / `thread_id` columns - not as a soft filter in Python. A bug in your harness can't leak Mercer's note to Vance; only a SQL injection could. For regulated infrastructure, add VPD policies on top (see `docs/part-5-scoping.md`).


# Part 6: The CityOps Copilot - End-to-end

> **Assembly.** Every component you built and verified in Parts 3–5 now snaps together into a single `call_copilot` turn - context card, similar-finding search, scoped reads, and the write-back that grows memory.


One function, `call_copilot`, ties together: thread resolution (SDK), asset lookup (`CITY_ASSET` SQL), the context card (SDK), similar-finding search (`CITY_INSPECTION_FINDING` SQL via `VECTOR_DISTANCE()`), OCI Generative AI LLM call, and persistence (which triggers `oracleagentmemory` auto-extraction).


In [ ]:
# Pre-built: system prompt used by OCI Generative AI chat.
COPILOT_SYSTEM_PROMPT = """You are a CityOps inspection copilot.

Each turn you are given:
- The current inspection narrative
- The asset record (class)
- A thread context card (recent inspector messages + extracted facts/guidelines)
- Up to 3 similar past findings on the same asset (with category, severity,
  recommendation, prior inspector, prior overall grade)

Your job: suggest a likely diagnosis or characterisation; cite prior findings
with their inspector + grade + recommendation timeline; surface relevant
guidelines from the thread context. Keep responses ≤ 8 sentences.
When safety- or maintenance-critical guidelines apply, name them."""

# call_oci_genai_chat(...) was defined in the SDK setup cell and is reused here
# for the copilot's visible model call.
ok(f"OCI Generative AI chat helper ready: {OCI_GENAI_CHAT_MODEL}")


### What does `get_context_card` actually return?

Before you wire it into `call_copilot` (TODO 4), see what the SDK gives you. The cell below grabs the `inspect_demo` thread (4 narratives from Part 3) and prints `card.formatted_content` verbatim.

The card is **XML, not Markdown**. Four blocks: `<summary>`, `<topics>`, `<relevant_information>`, `<recent_messages>`.


In [ ]:
_demo_thread = memory.get_thread("inspect_demo")
_demo_card = await _demo_thread.get_context_card_async(
    fallback_message_count=100,
    max_recent_messages=5,
    max_relevant_results=5,
)
print("=" * 70)
print("Raw context-card output - this is the XML the agent LLM will see:")
print("=" * 70)
print(_demo_card.formatted_content)

ok("Cell is complete")


Everything mechanical - thread resolution, asset lookup, context assembly, the LLM call, persistence - is already wired. **You write the two reads that pull in prior context:**

1. `card = await t.get_context_card_async(fallback_message_count=100, max_recent_messages=10, max_relevant_results=8)` - the SDK's conversational state
2. `similar = find_similar_findings(narrative, asset_id=asset_id, k=3)` - SQL-backed prior findings (sync; our own helper, not an SDK call)

These two are what let Vance's turn see Mercer's work with no human handoff.


In [ ]:
async def call_copilot(narrative: str, inspector_id: str, thread_id: str, asset_id: str) -> str:
    """End-to-end CityOps copilot turn: build context, query LLM, persist."""
    with timed("1 resolve thread"):
        try:
            t = memory.get_thread(thread_id)
        except Exception:
            t = None
        if t is None:
            t = memory.create_thread(
                user_id=inspector_id,
                thread_id=thread_id,
                agent_id="CITY",
                enable_context_summary=True,
                context_summary_update_frequency=2,
            )

    with timed("2 asset lookup"):
        asset = get_asset(asset_id)
        asset_info = (
            f"Asset {asset['asset_id']} (class: {asset['asset_class']})"
            if asset else "(no asset record found)"
        )

    with timed("3 context card (SDK memory search/summary)"):
        # TODO 4a: build the context card (assign to `card`).
        # card = await t.get_context_card_async(fallback_message_count=100,
        #     max_recent_messages=COPILOT_MAX_RECENT_MESSAGES,
        #     max_relevant_results=COPILOT_MAX_RELEVANT_RESULTS)
        # YOUR CODE HERE
        card = None

    with timed("4 similar findings (OCI query embed + Oracle vector search)"):
        # TODO 4b: find similar prior findings on this asset (assign to `similar`).
        # similar = find_similar_findings(narrative, asset_id=asset_id, k=3)
        # YOUR CODE HERE
        similar = []
    if similar:
        similar_text = "\n\n".join(
            f"  (score={r['score']:.3f})  [{r['category']}/{r['severity']}]  "
            f"inspector={r['inspector']}, grade={r['overall_grade']}, days_ago={r['days_ago']}\n"
            f"     description: {r['description']}\n"
            f"     recommendation: {r['recommendation']}"
            for r in similar
        )
    else:
        similar_text = "  (no prior findings for this asset)"

    context = (
        f"# Current inspection narrative\n"
        f"Asset: {asset_id}\n"
        f"Inspector: {inspector_id}\n"
        f"Narrative: {narrative}\n\n"
        f"# Asset record\n{asset_info}\n\n"
        f"# Thread context\n{card.formatted_content}\n\n"
        f"# Similar past findings (from CITY_INSPECTION_FINDING)\n{similar_text}"
    )

    print("=" * 70)
    print("USER MESSAGE sent to the model:")
    print("=" * 70)
    print(context)
    print("=" * 70 + "\n")

    messages = [
        {"role": "system", "content": COPILOT_SYSTEM_PROMPT},
        {"role": "user",   "content": context},
    ]
    with timed("5 OCI chat call"):
        resp = await call_oci_genai_chat(messages, max_tokens=COPILOT_MAX_TOKENS)
    answer = resp.choices[0].message.content or ""

    with timed("6 persist turn (SDK write + extraction)"):
        await t.add_messages_async([
            Message(role="user",      content=f"[{inspector_id} @ {asset_id}] {narrative}"),
            Message(role="assistant", content=answer),
        ])
    return answer

# Checkpoint: TODO 4 - smoke test
with timed("TODO 4 call_copilot total"):
    _smoke = await call_copilot(
        narrative="Smoke test - please ignore. One-line check on Harbor Bridge.",
        inspector_id="smoke_inspector",
        thread_id="copilot_smoke",
        asset_id="Harbor Bridge",
    )
assert _smoke and len(_smoke) > 10, "TODO 4 - copilot returned empty/short answer"
ok("TODO 4 passed - copilot ran end-to-end")
ok(f"\nSample response (first 300 chars):\n  {_smoke[:300]}")


<details>
<summary>💡 Show answer (TODO 4)</summary>

```python
# 6a - context card
card = await t.get_context_card_async(
    fallback_message_count=100,
    max_recent_messages=COPILOT_MAX_RECENT_MESSAGES,
    max_relevant_results=COPILOT_MAX_RELEVANT_RESULTS,
)
# 6b - similar prior findings
similar = find_similar_findings(narrative, asset_id=asset_id, k=3)
```

</details>

## The cross-inspector handoff scenario

Inspector Mercer reviews Harbor Bridge, logs a corrosion finding. Days later, Inspector Vance - who has never met Mercer - encounters a related concern on the same asset. Watch what the copilot tells Vance.


### Day 1 - Mercer's visit uses both functions, in sequence

The two functions you built in Parts 4 and 6 play **complementary roles** in a real inspection workflow. Mercer's day-1 visit shows both:

| Step | Function | Role | Cost |
|---|---|---|---|
| 1 | `call_copilot(...)` | Reasoning interface. Mercer types what she sees; the copilot interprets, suggests, surfaces any prior findings on this asset. | ~4 LLM calls, 30–60 s |
| 2 | `log_finding(...)` | System of record. Mercer decides what the finding is and stores it as a structured row with category, severity, recommendation, grade. | 0 LLM calls, 1 OCI embedding call |

**Why both?** `call_copilot` is the diagnostic dialogue; `log_finding` is the formal record. The structured row Mercer logs in step 2 becomes **searchable evidence** for the next inspector's `call_copilot` invocation - via `find_similar_findings` inside the copilot - closing the loop.

Below: each function gets its own cell.


In [ ]:
# Step 1 - Mercer reports what she sees and asks the copilot for help.
# This is the REASONING call: builds a context card, vector-searches
# CITY_INSPECTION_FINDING for prior issues on Harbor Bridge, and runs the
# agent LLM. On this first visit there are no prior findings, so the
# copilot's response is mostly fresh diagnosis.
print("=" * 70)
print("DAY 1, step 1: Mercer calls call_copilot for diagnostic help")
print("=" * 70)
MERCER_NOTES = await call_copilot(
    narrative=(
        "Quarterly inspection of Harbor Bridge. Surface corrosion observed on Pier 2 "
        "bearing assemblies (south side), estimated section loss ~25% on bearing plate "
        "edges with local rust bleeding onto the concrete pedestal. Corrosion extends "
        "roughly 1.5 m longitudinally along the bearing line. Standing guidance for Harbor Bridge: always remove loose corrosion products before applying inhibiting primer, and re-inspect Pier 2 bearings annually."
    ),
    inspector_id="Evelyn_H_Mercer",
    thread_id="asset_harbor_bridge",
    asset_id="Harbor Bridge",
)
print(f"\nMercer's copilot response:\n{MERCER_NOTES}")

ok("Cell is complete")


Now Mercer has decided this is a real corrosion concern. The copilot's answer helped her think; the structured fields are her conclusion.

**Step 2** is the recording call - `log_finding` writes one row into `CITY_INSPECTION_FINDING` with the embedding computed by OCI Generative AI on the `description`.

This row is what `find_similar_findings` will surface for the next inspector.


In [ ]:
# Step 2 - Mercer formally records the finding she's now confident about.
# Pure persistence: one INSERT + one OCI embedding.
MERCER_FINDING_ID = log_finding(
    asset_id="Harbor Bridge",
    inspector="Evelyn_H_Mercer",
    category="corrosion",
    severity="medium",
    description=(
        "Surface corrosion + pitting on steel bearing assemblies at Pier 2 south, "
        "~25% section loss; rust bleed onto concrete pedestal; ~1.5m longitudinal extent."
    ),
    recommendation=(
        "Remove loose corrosion products, apply corrosion-inhibiting primer + finish "
        "coat to affected bearing assemblies within 60 days; re-inspect annually with "
        "caliper section-loss measurements."
    ),
    overall_grade="C",
    days_ago=0,
)
print(f"Logged finding {MERCER_FINDING_ID} - now searchable by future call_copilot invocations.")

# Day 1, 14:00 - Mercer adds a follow-up observation
print("\n" + "=" * 70)
print("DAY 1 (afternoon): Mercer follow-up note")
print("=" * 70)
MERCER_FOLLOWUP = await call_copilot(
    narrative=(
        "Coordinated with maintenance - recommend scheduling the bearing remediation "
        "to coincide with the deck wearing-surface resurfacing in Q3 to share access "
        "and traffic management."
    ),
    inspector_id="Evelyn_H_Mercer",
    thread_id="asset_harbor_bridge",
    asset_id="Harbor Bridge",
)
print(f"\n Mercer's followup response:\n{MERCER_FOLLOWUP}")

ok("Cell is complete")


### Did Mercer's turn leave a `guideline`?

Her `call_copilot` turn ran the SDK extractor over her narrative. Scoped to the `asset_harbor_bridge` thread, `CITY_MEMORY` should now hold the **facts** she stated *plus* a **`[guideline]`** distilled from her standing-guidance line - the reusable rule Vance inherits next turn.

> Extraction is probabilistic: if the rule lands as a `fact` instead, just re-run Mercer's `call_copilot` cell.

In [ ]:
peek_sdk_tables(thread_id="asset_harbor_bridge")

### Day N - Vance only calls `call_copilot`. Why?

Vance is in the diagnostic phase - he's seen rust bleed and possible spalling, but he hasn't decided what the finding is yet. He calls `call_copilot` to **interpret** what he's seeing, not to record anything.

Under the hood, `call_copilot` does two things that pull in Mercer's prior work without anyone telling it to:

1. **`find_similar_findings(...)`** vector-searches `CITY_INSPECTION_FINDING` for Harbor Bridge - Mercer's logged finding (the row from the cell above) is the top hit. The copilot's prompt gets her category, severity, recommendation, and grade as structured fields.
2. **`thread.get_context_card(...)`** assembles Mercer's earlier messages + the facts and **guidelines** the SDK auto-extracted from them. So Vance inherits more than Mercer's observations - he gets her **operating rules**, like the standing guidance to *remove loose corrosion before priming* and *re-inspect Pier 2 bearings annually*. That's a `guideline` (what to *do*) versus a `fact` (what *is*) - and the formal finding carries neither.

If Vance later confirms his diagnosis, he'd add a step-2 `log_finding(...)` of his own - but the workshop stops here because the magic moment is Vance's reasoning being shaped by Mercer's prior work without human handoff.


In [ ]:
# Day N (later) - Inspector Vance arrives. Different inspector, same asset.
# Only call_copilot - Vance is still in diagnosis mode, not yet recording.
print("\n" + "=" * 70)
print("DAY N: Inspector Vance on Harbor Bridge (never met Mercer)")
print("=" * 70)
VANCE_DIAGNOSIS = await call_copilot(
    narrative=(
        "Reviewing Harbor Bridge as part of routine cycle. Noticing rust bleed near "
        "a pier on the south side and what looks like spalling on the concrete "
        "pedestal below."
    ),
    inspector_id="Jordan_Vance",
    thread_id="asset_harbor_bridge",
    asset_id="Harbor Bridge",
)
print(f"\nVance's copilot response (built from Mercer's work, NO human handoff):\n{VANCE_DIAGNOSIS}")

ok("Cell is complete")


## Compare: Vance with and without memory

Below: same Vance narrative, but stripped of all memory layers - no thread, no context card, no `find_similar_findings`. Just the LLM and the narrative. The contrast is the workshop's point.


In [ ]:
stateless_messages = [
    {"role": "system", "content": COPILOT_SYSTEM_PROMPT},
    {"role": "user",   "content": (
        "Reviewing Harbor Bridge as part of routine cycle. Noticing rust bleed near "
        "a pier on the south side and what looks like spalling on the concrete "
        "pedestal below."
    )},
]
STATELESS_VANCE = (await call_oci_genai_chat(stateless_messages)).choices[0].message.content

print("=" * 70)
print("WITHOUT MEMORY (stateless LLM):")
print("=" * 70)
print(STATELESS_VANCE)
print("\n" + "=" * 70)
print("WITH MEMORY (the copilot you built):")
print("=" * 70)
print(VANCE_DIAGNOSIS)

ok("Cell is complete")


## Key takeaways

| Layer | Earned its keep by… |
|---|---|
| Auto-extracted `fact` / `preference` / `guideline` (SDK) | Surfacing tribal knowledge - facts (what *is*) and guidelines (what to *do*) - from real maintenance narratives without explicit code |
| `CITY_ASSET` SQL table | Letting the copilot look up structured asset facts at the start of every turn |
| `CITY_INSPECTION_FINDING` SQL with `VECTOR(OCI_EMBED_DIM)` | Vector-searchable history via Oracle's native `VECTOR_DISTANCE()` - mixed with relational filters in one SQL |
| Context card (SDK) | Compressing the per-asset thread into ~200 tokens that travel with every turn |
| Scoping (`user_id` / `agent_id` / `thread_id`) | Multi-tenant-safety at the SQL layer |


## Memory types - what you actually built

It's easy to miss how many of the standard agent-memory types you implemented with `oracleagentmemory`. Here's the connection back.

**The four SDK record types map onto the cognitive memory types:**

| SDK `record_type` | Cognitive type | Example from this lab |
|---|---|---|
| `fact` | **Semantic** - declarative knowledge | *"Harbor Bridge main span is 485 m"* |
| `preference` | **Persona** - stable choices / styles | an inspector's defaults |
| `guideline` | **Procedural** - reusable rules / "next time do X" | *"remove corrosion before priming; re-inspect Pier 2 annually"* |
| `memory` | **Working / ongoing state** - the fallback bucket | reminders, decisions, and anything written via `add_memory` |

**And the rest of the agent-memory landscape:**

| Standard memory type | What you used here |
|---|---|
| Working memory / **LLM context window** | `get_context_card()` - the prompt-ready block |
| **Session / short-term** | the thread (`CITY_THREAD`) + its recent messages |
| Episodic - **conversations** | `CITY_MESSAGE` - verbatim turns per thread |
| Episodic - **summaries** | the rolling thread summary (`enable_context_summary`) |
| Semantic - **knowledge base** | `fact` records + `VECTOR_DISTANCE()` search |
| Semantic - **persona** | `preference` records + actor profiles |
| **Shared / coordination** | `agent_id`-scoped memory (the city-wide note every inspector sees) |

**Left to the agent *framework* (e.g. LangGraph), not the memory layer:** Semantic Cache, and the tool-orchestration artifacts - Toolbox, Workflow, Tool Logs.

> **One line:** `oracleagentmemory` is the *governed memory core* - it covers what the agent **knows, said, learned, and shares** (semantic, episodic, working, persona, shared). Tool/workflow orchestration and response caching are the framework's job.

## Where to next?

- **[Oracle AI Agent Memory documentation](https://docs.oracle.com/en/database/oracle/agent-memory/)**
- **[Oracle AI Developer Hub](https://github.com/oracle-devrel/oracle-ai-developer-hub)**
- **[Agent Memory short course (DeepLearning.AI)](https://www.deeplearning.ai/short-courses/agent-memory-building-memory-aware-agents/)**
